# Appliance Energy Forecasting — Part 2 & 3: Problem Definition & Benchmark Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/appliance-energy-forecasting"
os.chdir(PROJECT_ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (14, 5)
np.random.seed(0)

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/forecasts", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)


In [ ]:
# load cleaned hourly data produced in notebook 1
hourly = pd.read_csv("data/processed/appliance_hourly.csv", index_col=0, parse_dates=True)
y = hourly["Appliances"]
print(y.shape)
y.head()


In [ ]:
# target variable: Appliances (Wh)
TARGET = "Appliances"

# forecast horizon: 24 hours ahead
HORIZON = 24

# seasonal periods for hourly data
DAILY_PERIOD = 24
WEEKLY_PERIOD = 168

# test set: final 14 days
TEST_STEPS = 14 * 24

train = y.iloc[:-TEST_STEPS]
test = y.iloc[-TEST_STEPS:]

print("train period:", train.index.min(), "to", train.index.max(), "n =", len(train))
print("test period:", test.index.min(), "to", test.index.max(), "n =", len(test))


In [ ]:
# evaluation metrics: MAE, RMSE, MASE, Bias
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mase(y_true, y_pred, y_train, seasonality=24):
    # scale = mean absolute error of in-sample seasonal naive forecast
    y_train = pd.Series(y_train).astype(float)
    seasonal_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = seasonal_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

def evaluate_forecast(name, y_true, y_pred, y_train):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD),
        "Bias": np.mean(y_pred - y_true),
    }


In [ ]:
# mean forecast: repeats the training mean for every step
def mean_forecast(y_train, horizon, index):
    return pd.Series(y_train.mean(), index=index, name="mean")

# naive forecast: repeats the last observed training value
def naive_forecast(y_train, horizon, index):
    return pd.Series(y_train.iloc[-1], index=index, name="naive")

# seasonal naive forecast: repeats the value from `seasonality` steps ago, recursively
def seasonal_naive_forecast(y_train, horizon, index, seasonality):
    values = []
    history = list(y_train.values)
    for i in range(horizon):
        values.append(history[-seasonality])
        history.append(values[-1])
    return pd.Series(values, index=index)

# drift forecast: extrapolates the average slope across the training set
def drift_forecast(y_train, horizon, index):
    slope = (y_train.iloc[-1] - y_train.iloc[0]) / (len(y_train) - 1)
    values = [y_train.iloc[-1] + slope * step for step in range(1, horizon + 1)]
    return pd.Series(values, index=index, name="drift")


In [ ]:
# generate forecasts over the full test period, using a rolling 24h window
# so each 24h block is forecast from the true history up to that point
def rolling_forecast(forecast_fn, y_train, y_test, horizon=24, **kwargs):
    history = y_train.copy()
    preds = []
    for start in range(0, len(y_test), horizon):
        block_index = y_test.index[start:start + horizon]
        block_horizon = len(block_index)
        pred = forecast_fn(history, block_horizon, block_index, **kwargs)
        preds.append(pred)
        history = pd.concat([history, y_test.iloc[start:start + horizon]])
    return pd.concat(preds)

forecasts = {}
forecasts["mean"] = rolling_forecast(mean_forecast, train, test, horizon=HORIZON)
forecasts["naive"] = rolling_forecast(naive_forecast, train, test, horizon=HORIZON)
forecasts["seasonal_naive_daily"] = rolling_forecast(
    seasonal_naive_forecast, train, test, horizon=HORIZON, seasonality=DAILY_PERIOD
)
forecasts["seasonal_naive_weekly"] = rolling_forecast(
    seasonal_naive_forecast, train, test, horizon=HORIZON, seasonality=WEEKLY_PERIOD
)
forecasts["drift"] = rolling_forecast(drift_forecast, train, test, horizon=HORIZON)


In [ ]:
# evaluate all benchmark forecasts against the test set
results = []
for name, pred in forecasts.items():
    pred = pred.reindex(test.index)
    results.append(evaluate_forecast(name, test, pred, train))

benchmark_results = pd.DataFrame(results).sort_values("MASE").reset_index(drop=True)
print(benchmark_results.round(3))

benchmark_results.to_csv("outputs/metrics/benchmark_comparison.csv", index=False)


In [ ]:
# plot benchmark forecasts against actual test data, first 7 days of test period for clarity
fig, ax = plt.subplots(figsize=(14, 7))
plot_window = test.index[:24*7]

test.loc[plot_window].plot(ax=ax, label="actual", color="black", linewidth=2)
for name, pred in forecasts.items():
    pred.reindex(plot_window).plot(ax=ax, label=name, alpha=0.8)

ax.set_title("Benchmark Forecasts vs Actual - First 7 Days of Test Period")
ax.set_ylabel("Appliances (Wh)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/06_benchmark_forecasts.png", dpi=150)
plt.show()


In [ ]:
# save all benchmark forecasts for later comparison against SARIMAX, ML, and foundation models
forecast_df = pd.DataFrame({"actual": test})
for name, pred in forecasts.items():
    forecast_df[name] = pred.reindex(test.index)

forecast_df.to_csv("outputs/forecasts/benchmark_forecasts.csv")
print("saved to outputs/forecasts/benchmark_forecasts.csv")
